# 06 – Baseline forecasts

The baseline is the reference against which the models are scored (notebook 09, relative WIS = WIS of the model / WIS of the baseline). It does not use any model: it predicts as median the last observed value and takes the width of the intervals from the past weekly increments (symmetrised random walk, 10,000 trajectories; Gozzi, Perra, Vespignani, *PNAS* 122(24), 2025, Materials and Methods). See `pinnvsmech.forecasting.baseline`.

It depends only on the observed series, so it is generated once for all the cases:

| Case | Observed series | Windows (current week t0) |
|---|---|---|
| real London | weekly deaths | rolling windows t0 = 3 … 22 |
| synthetic R0 = 1.5 (light / mild noise) | **noisy** weekly infections (the ones the models are calibrated on) | t0 = 3 … 29 and the extra window t0 = 33 |
| synthetic R0 = 2.0 (light / mild noise) | **noisy** weekly infections | t0 = 3 … 19 and the extra window t0 = 23 |

The windows are those of `case.t0_estimation()`, the same of the PINN and of the Mechanistic forecasts (notebooks 03–05). The baseline of a window covers t = t0 … t0 + HORIZON (there is no fit).

Output: `outputs/baselines/<case>/forecast_results_baseline.pkl`, in the format of the forecasts of the models (copy it to `precomputed/baselines/` to keep it). `<case>` is the label of the case without the look-back (the baseline does not use it), e.g. `real_London`, `synthetic_R0_1.5_noise_light`.

In [ ]:
import os
import pickle

import numpy as np

from pinnvsmech.forecasting.baseline import baseline_label, compute_baseline_forecasts_from_series
from pinnvsmech.inference.cases import load_case
from pinnvsmech.paths import OUTPUTS_DIR

## Settings

In [ ]:
# Cases to generate: "all" (real + the four synthetic scenarios) or a list of (kind, R0, noise)
CASES = "all"
# CASES = [("real", None, None), ("synthetic", 1.5, "light")]

HORIZON = 4                        # weeks ahead (H in the paper)
N_TRAJECTORIES = 10000             # bootstrap trajectories (as in the paper)
SEED = 0                           # None: not reproducible
CLIP_NEGATIVE = True               # weekly counts cannot be negative: clip the trajectories at 0 (not in the paper)

OUT_DIR = OUTPUTS_DIR / "baselines"

## Generate

In [ ]:
ALL_CASES = [("real", None, None)] + [("synthetic", R0, noise) for R0 in (1.5, 2.0) for noise in ("light", "mild")]
cases = ALL_CASES if CASES == "all" else CASES

for kind, R0, noise in cases:
    case = load_case(kind, R0=R0, noise=noise)
    t0_list = case.t0_estimation()

    baseline = compute_baseline_forecasts_from_series(
        case.obs, t0_list, dates=case.dates, horizon=HORIZON, n_trajectories=N_TRAJECTORIES, seed=SEED,
        clip_negative=CLIP_NEGATIVE,
    )

    out_dir = OUT_DIR / baseline_label(case)
    os.makedirs(out_dir, exist_ok=True)
    with open(out_dir / "forecast_results_baseline.pkl", "wb") as f:
        pickle.dump(baseline, f)
    print(f"{baseline_label(case):32s} {len(baseline)} windows, t0 {min(t0_list)}..{max(t0_list)} -> {out_dir}")